<a href="https://colab.research.google.com/github/14marcos1/ELT578/blob/main/desafio_placafinal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import cv2
import os

path = "PLACA5L.jpg"   # troque aqui pelo caminho real, se necessário
if not os.path.exists(path):
    raise FileNotFoundError(f"Arquivo não encontrado: {path}")

img = cv2.imread(path)
if img is None:
    raise ValueError("OpenCV não conseguiu ler a imagem.")

print(img.shape)


(780, 1040, 3)


In [10]:
import os
print(os.getcwd())
print(os.listdir("."))


/content
['.config', 'edges.png', 'placa_overlay.png', 'overlay.png', 'placa_retificada.png', 'PLACA5L.jpg', 'sample_data']


In [11]:
import cv2
import numpy as np
import os
import glob

def find_image():
    candidates = [
        "PLACA5L.jpg",
        "/content/PLACA5L.jpg",
        "/mnt/data/PLACA5L.jpg",
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    jpgs = glob.glob("*.jpg") + glob.glob("/content/*.jpg") + glob.glob("/mnt/data/*.jpg")
    if jpgs:
        return jpgs[0]
    raise FileNotFoundError("Nenhuma imagem JPG encontrada.")

def order_points(pts):
    pts = np.array(pts, dtype="float32")
    s = pts.sum(axis=1)
    diff = np.diff(pts, axis=1)
    return np.array([
        pts[np.argmin(s)],
        pts[np.argmin(diff)],
        pts[np.argmax(s)],
        pts[np.argmax(diff)],
    ], dtype="float32")

def detect_plate_quad(image_bgr):
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    gray = cv2.bilateralFilter(gray, 11, 17, 17)
    edges = cv2.Canny(gray, 30, 120)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    edges = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, kernel, iterations=3)

    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contours = sorted(contours, key=cv2.contourArea, reverse=True)

    best = None
    for c in contours[:50]:
        peri = cv2.arcLength(c, True)
        for eps in [0.02, 0.03, 0.04, 0.05]:
            approx = cv2.approxPolyDP(c, eps * peri, True)
            area = cv2.contourArea(approx)
            if len(approx) == 4 and area > 200:
                best = approx.reshape(4, 2)
                break
        if best is not None:
            break

    if best is None:
        return None, None, edges, None

    rect = order_points(best)
    (tl, tr, br, bl) = rect

    maxWidth = int(max(np.linalg.norm(br - bl), np.linalg.norm(tr - tl)))
    maxHeight = int(max(np.linalg.norm(tr - br), np.linalg.norm(tl - bl)))

    dst = np.array([
        [0, 0],
        [maxWidth - 1, 0],
        [maxWidth - 1, maxHeight - 1],
        [0, maxHeight - 1]
    ], dtype="float32")

    M = cv2.getPerspectiveTransform(rect, dst)
    warped = cv2.warpPerspective(image_bgr, M, (maxWidth, maxHeight))

    overlay = image_bgr.copy()
    cv2.polylines(overlay, [best.astype(int)], True, (0, 255, 0), 2)
    return warped, overlay, edges, rect

path = find_image()
img = cv2.imread(path)
if img is None:
    raise ValueError(f"Falha ao ler: {path}")

plate, overlay, edges, pts = detect_plate_quad(img)

if plate is None:
    raise RuntimeError("Não foi possível detectar um quadrilátero. Tente o modo Hough/YOLO.")

cv2.imwrite("overlay.png", overlay)
cv2.imwrite("edges.png", edges)
cv2.imwrite("placa_retificada.png", plate)

print("arquivo:", path)
print("pontos:", pts)


arquivo: PLACA5L.jpg
pontos: [[235. 487.]
 [367. 506.]
 [347. 565.]
 [247. 552.]]


In [12]:
import cv2
import numpy as np
import os

path = "PLACA5L.jpg"
img = cv2.imread(path)
if img is None:
    raise FileNotFoundError(f"Não consegui abrir {path}")

src_pts = np.array([
    [235, 487],
    [367, 506],
    [347, 565],
    [247, 552]
], dtype=np.float32)

def order_points(pts):
    pts = np.array(pts, dtype="float32")
    s = pts.sum(axis=1)
    diff = np.diff(pts, axis=1)
    tl = pts[np.argmin(s)]
    br = pts[np.argmax(s)]
    tr = pts[np.argmin(diff)]
    bl = pts[np.argmax(diff)]
    return np.array([tl, tr, br, bl], dtype="float32")

rect = order_points(src_pts)
(tl, tr, br, bl) = rect

widthA = np.linalg.norm(br - bl)
widthB = np.linalg.norm(tr - tl)
maxWidth = int(max(widthA, widthB))

heightA = np.linalg.norm(tr - br)
heightB = np.linalg.norm(tl - bl)
maxHeight = int(max(heightA, heightB))

dst_pts = np.array([
    [0, 0],
    [maxWidth - 1, 0],
    [maxWidth - 1, maxHeight - 1],
    [0, maxHeight - 1]
], dtype=np.float32)

M = cv2.getPerspectiveTransform(rect, dst_pts)
warped = cv2.warpPerspective(img, M, (maxWidth, maxHeight))

overlay = img.copy()
cv2.polylines(overlay, [rect.astype(int)], True, (0, 255, 0), 2)
for i, (x, y) in enumerate(rect.astype(int), 1):
    cv2.circle(overlay, (x, y), 5, (0, 0, 255), -1)
    cv2.putText(overlay, str(i), (x+5, y-5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)

cv2.imwrite("placa_overlay.png", overlay)
cv2.imwrite("placa_retificada.png", warped)

print("Pontos ordenados:", rect)
print("Salvo: placa_overlay.png e placa_retificada.png")


Pontos ordenados: [[235. 487.]
 [367. 506.]
 [347. 565.]
 [247. 552.]]
Salvo: placa_overlay.png e placa_retificada.png


In [15]:
!apt install tesseract-ocr tesseract-ocr-por
!pip install pytesseract


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
The following NEW packages will be installed:
  tesseract-ocr-por
0 upgraded, 1 newly installed, 0 to remove and 6 not upgraded.
Need to get 856 kB of archives.
After this operation, 1,998 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tesseract-ocr-por all 1:4.00~git30-7274cfa-1.1 [856 kB]
Fetched 856 kB in 1s (758 kB/s)
Selecting previously unselected package tesseract-ocr-por.
(Reading database ... 118194 files and directories currently installed.)
Preparing to unpack .../tesseract-ocr-por_1%3a4.00~git30-7274cfa-1.1_all.deb ...
Unpacking tesseract-ocr-por (1:4.00~git30-7274cfa-1.1) ...
Setting up tesseract-ocr-por (1:4.00~git30-7274cfa-1.1) ...


In [16]:
# Versão manual com EasyOCR (reinicia o runtime antes)
!pip uninstall easyocr -y
!pip install easyocr


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.6/300.6 kB 21.5 MB/s eta 0:00:00


In [18]:
import cv2
import numpy as np
import pytesseract
import os

def preprocess_placa(plate_bgr):
    gray = cv2.cvtColor(plate_bgr, cv2.COLOR_BGR2GRAY)

    # CLAHE para contraste
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
    gray = clahe.apply(gray)

    # Desfoque para suavizar ruído
    gray = cv2.GaussianBlur(gray, (1, 1), 0)

    # Threshold adaptativo
    thresh = cv2.adaptiveThreshold(
        gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY, 31, 10
    )

    # Morph para conectar letras
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)

    # Redimensiona para OCR (melhora muito!)
    h, w = thresh.shape
    new_w = int(w * 3)  # 3x mais largo
    thresh = cv2.resize(thresh, (new_w, int(h * 3)))

    return thresh

# Carregue sua placa retificada
img = cv2.imread('placa_retificada.png')
if img is None:
    raise FileNotFoundError("placa_retificada.png não encontrada.")

# Pré-processa
thresh = preprocess_placa(img)

# Config Tesseract para placas
custom_config = r'--oem 3 --psm 8 -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789'

# Tenta múltiplas configurações
configs = [
    custom_config,
    r'--oem 3 --psm 8',
    r'--oem 3 --psm 7',
    r'--oem 3 --psm 13'
]

for i, config in enumerate(configs):
    text = pytesseract.image_to_string(thresh, config=config)
    text = ''.join(c for c in text.upper() if c.isalnum()).strip()
    if len(text) >= 6:
        print(f"Config {i}: {text}")
        break
else:
    print("Tentativa final:", pytesseract.image_to_string(thresh))

cv2.imwrite("placa_preprocessada.png", thresh)
print("Salvou placa_preprocessada.png")


Config 0: FEAIBV
Salvou placa_preprocessada.png


In [28]:
# FUNÇÕES que FUNCIONARAM nas suas células
def order_points(pts):
    pts
